# Projet 5 - Resumeur controlable

**Track A (Ollama local).** Resumer une transcription de reunion en controlant
l'audience, la longueur et le format, puis en extraire les decisions a prendre.

- **Donnees :** `data/transcripts.jsonl` - 8 transcriptions, chacune avec un
  `reference_summary` et une liste `action_items` de reference.
- **Evaluation :** controles automatiques longueur/format, juge LLM de fidelite
  (valide, pas pris pour argent comptant), et rappel des `action_items`.

> Les cellules de code et les prompts sont en anglais (le modele y est plus stable,
> comme les helpers du cours) ; les commentaires d'analyse sont en francais.

## 1. Configuration

In [1]:
# Two ways to reach a model, tried in order:
#   1. the course helpers (tiktoken token counts, temperature control) if the
#      langchain_courses repo is cloned next to this one;
#   2. otherwise the local utils.py, which calls Ollama directly and keeps this
#      repo self-contained.
import importlib
import json
import re
import statistics
import sys
from pathlib import Path

HERE = Path.cwd()
COURSE = next(
    (p / "langchain_courses" / "prompt-engineering-course"
     for p in [HERE, *HERE.parents]
     if (p / "langchain_courses" / "prompt-engineering-course").is_dir()),
    None,
)

if COURSE is not None:
    sys.path.insert(0, str(COURSE))
    for name in [m for m in sys.modules if m == "utils" or m.startswith("utils.")]:
        del sys.modules[name]          # drop the local utils.py if it was imported first
    _utils = importlib.import_module("utils")
    ask, count_tokens = _utils.ask, _utils.count_tokens
    MODEL = _utils.SMALL_MODEL
    BACKEND = "helpers du cours"
else:
    import utils as _utils            # local fallback (utils.py at the repo root)

    MODEL = "llama3.2"
    BACKEND = "utils.py local"

    def ask(prompt, temperature=0.0, model=MODEL):
        """Adapter: the local helper takes no temperature, so it is ignored here."""
        return _utils.ask(prompt, model=model)

    count_tokens = _utils.count_tokens

MAX_WORDS = 50           # length budget enforced by the checks below
TEMPERATURE = 0.0        # deterministic: the evaluation must be reproducible

DOCS = [json.loads(line) for line in open("data/transcripts.jsonl", encoding="utf-8")]
print(f"backend={BACKEND} | model={MODEL} | {len(DOCS)} transcripts | fields={list(DOCS[0])}")

backend=helpers du cours | model=llama3.2:3b | 8 transcripts | fields=['text', 'reference_summary', 'action_items']


## 2. Apercu des donnees

On verifie ce que contient reellement le jeu de donnees avant de prompter : la
taille des textes conditionne le budget de tokens, et le champ `action_items`
fournit une verite terrain qu'on exploitera en section 8.

In [2]:
for i, doc in enumerate(DOCS, 1):
    print(f"[{i}] {count_tokens(doc['text']):>3} tok"
          f" | {len(doc['action_items'])} action item(s)"
          f" | {doc['text'][:60]}...")

total = sum(count_tokens(d["text"]) for d in DOCS)
print(f"\nTotal corpus: {total} tokens (~{total / len(DOCS):.0f} par transcription)")

[1]  53 tok | 1 action item(s) | The product team reviewed the mobile checkout redesign. The ...
[2]  50 tok | 2 action item(s) | Operations reported that the warehouse migration is on sched...
[3]  50 tok | 2 action item(s) | The marketing team analyzed the spring campaign. Email open ...
[4]  42 tok | 2 action item(s) | Customer support reviewed the top complaints from April. Cus...
[5]  49 tok | 1 action item(s) | The research group presented results from the onboarding exp...
[6]  44 tok | 2 action item(s) | Finance reviewed the quarterly forecast. Revenue is slightly...
[7]  45 tok | 1 action item(s) | The security team completed the access review. Most permissi...
[8]  46 tok | 1 action item(s) | The leadership team discussed the remote-work policy. Employ...

Total corpus: 379 tokens (~47 par transcription)


## 3. Conception du prompt

Le prompt de depart demandait  3 bullets  et une section `Actions:` sans jamais
verifier que le modele obeissait. On rend donc le **contrat de format explicite**
et surtout **verifiable** : un petit modele a besoin de contraintes nettes, et ce
qui n'est pas mesure n'est pas respecte.

In [3]:
PROMPT = """Summarize the meeting transcript below.

Audience: {audience}
Hard limits:
- At most {max_words} words in total.
- Exactly 3 bullet points, each starting with "- ".
- Then a line containing exactly "Actions:" followed by one line per action item,
  each starting with "- ". Write "- none" if there are no action items.

Return nothing else: no preamble, no title, no closing remark.

Transcript:
{text}"""


def summarize(text, audience="a non-technical manager", max_words=MAX_WORDS):
    """One summary, deterministic, under an explicit and checkable format contract."""
    return ask(
        PROMPT.format(audience=audience, max_words=max_words, text=text),
        temperature=TEMPERATURE,
        model=MODEL,
    )


print(summarize(DOCS[0]["text"]))
print("\n--- reference ---")
print(DOCS[0]["reference_summary"])

- The product team reviewed the mobile checkout redesign.
- The new payment form reduced the number of fields from nine to five.
- Accessibility testing found that screen readers do not announce validation errors.

Actions:
- Fix error announcements before beta release next Friday.
- - none

--- reference ---
The checkout redesign simplifies payment entry, but screen-reader validation errors must be fixed before next Friday's beta.


## 4. Tache 1 - Controles automatiques de longueur et de format

On genere les 8 resumes **une seule fois** et on les reutilise ensuite. Le starter
appelait `summarize()` trois fois sur les memes documents : cela triplait le temps
d'execution et, surtout, evaluait a chaque etape un texte different de celui qu'on
venait de juger.

In [4]:
SUMMARIES = [summarize(doc["text"]) for doc in DOCS]   # generated once, reused everywhere


def _strip_bullet(line):
    """Remove the leading dashes a small model sometimes doubles up ("- - none")."""
    return re.sub(r"^(?:[-*]\s+)+", "", line.strip()).strip()


def parse_summary(summary):
    """Split a summary into its bullet list and its action list."""
    head, _, tail = summary.partition("Actions:")
    bullets = [l.strip() for l in head.strip().splitlines() if l.strip().startswith(("- ", "* "))]
    actions = [_strip_bullet(l) for l in tail.strip().splitlines()
               if l.strip().startswith(("- ", "* "))]
    actions = [a for a in actions if a and a.lower().rstrip(".") != "none"]
    return bullets, actions, ("Actions:" in summary)


def check(summary):
    bullets, _, has_actions = parse_summary(summary)
    words = len(summary.split())
    return {
        "words": words,
        "length_ok": words <= MAX_WORDS,
        "bullets": len(bullets),
        "bullets_ok": len(bullets) == 3,
        "actions_ok": has_actions,
    }


checks = [check(s) for s in SUMMARIES]

print(f"{'doc':<5}{'mots':>6}{'len':>5}{'bul':>5}{'fmt':>5}{'act':>5}   verdict")
for i, c in enumerate(checks, 1):
    ok = c["length_ok"] and c["bullets_ok"] and c["actions_ok"]
    print(f"{i:<5}{c['words']:>6}{'OK' if c['length_ok'] else 'X':>5}"
          f"{c['bullets']:>5}{'OK' if c['bullets_ok'] else 'X':>5}"
          f"{'OK' if c['actions_ok'] else 'X':>5}   {'PASS' if ok else 'FAIL'}")


def rate(key):
    return sum(c[key] for c in checks) / len(checks)


print(f"\nlongueur <= {MAX_WORDS} mots : {rate('length_ok'):.0%}")
print(f"exactement 3 bullets     : {rate('bullets_ok'):.0%}")
print(f"section Actions presente : {rate('actions_ok'):.0%}")
print(f"mediane des longueurs    : {statistics.median(c['words'] for c in checks):.0f} mots")

doc    mots  len  bul  fmt  act   verdict
1        48   OK    3   OK   OK   PASS
2        63    X    3   OK   OK   FAIL
3        51    X    3   OK   OK   FAIL
4        64    X    3   OK   OK   FAIL
5        46   OK    3   OK   OK   PASS
6        46   OK    3   OK   OK   PASS
7        62    X    3   OK   OK   FAIL
8        58    X    3   OK   OK   FAIL

longueur <= 50 mots : 38%
exactement 3 bullets     : 100%
section Actions presente : 100%
mediane des longueurs    : 54 mots


## 5. Un juge LLM robuste

Le juge du starter faisait `json.loads(raw)` directement. Sur un modele 3B la sortie
est frequemment entouree de texte ou d'un bloc ```json : le parsing echouait et le
score tombait silencieusement a `None`, ce qui donnait l'illusion d'une evaluation.
On extrait donc le premier objet JSON de la reponse, avec une seconde tentative.

In [5]:
JUDGE_PROMPT = """Rate how faithful the SUMMARY is to the SOURCE.

5 = every claim is supported by the source
3 = mostly supported, one questionable claim
1 = contains claims absent from or contradicting the source

Return ONLY this JSON object and nothing else:
{{"score": <integer 1-5>, "reason": "<one short sentence>"}}

SOURCE:
{source}

SUMMARY:
{summary}"""


def _extract_json(raw):
    match = re.search(r"\{.*?\}", raw, re.DOTALL)
    if not match:
        return None
    try:
        obj = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    return obj if isinstance(obj.get("score"), int) else None


def judge(summary, source, retries=1):
    """Faithfulness score 1-5, tolerant to a small model's formatting noise."""
    prompt = JUDGE_PROMPT.format(source=source, summary=summary)
    raw = ""
    for _ in range(retries + 1):
        raw = ask(prompt, temperature=TEMPERATURE, model=MODEL)
        parsed = _extract_json(raw)
        if parsed:
            return {"score": parsed["score"], "reason": str(parsed.get("reason", ""))[:120]}
        prompt = (JUDGE_PROMPT.format(source=source, summary=summary)
                  + "\n\nYour previous answer was not valid JSON."
                    " Return ONLY the JSON object.")
    return {"score": None, "reason": f"unparseable: {raw[:60]}"}


print(judge(SUMMARIES[0], DOCS[0]["text"]))

{'score': 4, 'reason': 'Most claims are supported, with only one minor omission of a specific action timeline'}


## 6. Tache 2 - Valider le juge

Un juge non valide ne prouve rien. On le controle de deux facons complementaires.

**a) Test contradictoire (objectif, automatique).** On lui soumet un resume fidele
et le meme resume volontairement falsifie. Un juge calibre doit nettement mieux
noter le premier. Ce test ne demande aucune annotation humaine.

**b) Accord avec nos propres notes.** La consigne demande de noter nous-memes 5
resumes. Ces notes doivent etre les notres : la cellule affiche les 5 resumes, on
inscrit nos scores dans `MANUAL_SCORES`, puis on mesure l'ecart avec le juge. Tant
que la liste contient des `None`, le notebook le signale au lieu de fabriquer un
faux accord.

In [6]:
# (a) Adversarial control: the same summary, deliberately corrupted.
faithful = SUMMARIES[0]
corrupted = (faithful
             + "\n- The team also approved a 2 million euro budget increase."
             + "\n- The CEO resigned during the meeting.")

score_faithful = judge(faithful, DOCS[0]["text"])["score"]
score_corrupted = judge(corrupted, DOCS[0]["text"])["score"]

print(f"resume fidele   -> {score_faithful}")
print(f"resume falsifie -> {score_corrupted}")

if None in (score_faithful, score_corrupted):
    print("VERDICT: juge inutilisable (aucun score exploitable)")
elif score_faithful > score_corrupted:
    print(f"VERDICT: juge discriminant (ecart de {score_faithful - score_corrupted} point(s))")
else:
    print("VERDICT: juge NON calibre - il ne detecte pas les affirmations inventees")

resume fidele   -> 4
resume falsifie -> 2
VERDICT: juge discriminant (ecart de 2 point(s))


In [7]:
# (b) Read the 5 summaries below, then fill MANUAL_SCORES in the next cell.
for i in range(5):
    print(f"\n===== Document {i + 1} =====")
    print("SOURCE :", DOCS[i]["text"][:200], "...")
    print("--- resume genere ---")
    print(SUMMARIES[i])


===== Document 1 =====
SOURCE : The product team reviewed the mobile checkout redesign. The new payment form reduced the number of fields from nine to five in testing, but accessibility testing found that screen readers do not annou ...
--- resume genere ---
- The product team reviewed the mobile checkout redesign.
- The new payment form reduced the number of fields from nine to five.
- Accessibility testing found that screen readers do not announce validation errors.

Actions:
- Fix error announcements before beta release next Friday.
- - none

===== Document 2 =====
SOURCE : Operations reported that the warehouse migration is on schedule. Inventory data has been copied, but barcode scanners in zone B still lose connection during peak traffic. The vendor will send a firmwa ...
--- resume genere ---
- Barcode scanners in zone B still experience connection issues during peak traffic.
- Inventory data has been successfully copied.
- The vendor will send a firmware patch on Tuesday.

Act

In [8]:
# Nos notes de fidelite (1-5), une par document, a remplir apres lecture ci-dessus.
MANUAL_SCORES = [None, None, None, None, None]

if any(s is None for s in MANUAL_SCORES):
    print("A COMPLETER : renseigner MANUAL_SCORES apres avoir lu les 5 resumes.")
    print("Tant que la liste n'est pas remplie, aucun accord juge/humain n'est calcule.")
else:
    rows = []
    for i, manual in enumerate(MANUAL_SCORES):
        auto = judge(SUMMARIES[i], DOCS[i]["text"])["score"]
        rows.append((i + 1, manual, auto, None if auto is None else abs(auto - manual)))

    print(f"{'doc':<5}{'nous':>6}{'juge':>6}{'ecart':>7}")
    for doc, manual, auto, delta in rows:
        print(f"{doc:<5}{manual:>6}{str(auto):>6}{str(delta):>7}")

    usable = [r for r in rows if r[3] is not None]
    if usable:
        mae = sum(r[3] for r in usable) / len(usable)
        exact = sum(r[3] == 0 for r in usable)
        print(f"\nErreur absolue moyenne : {mae:.2f} point(s)")
        print(f"Accord exact           : {exact}/{len(usable)}")
        print("Juge exploitable." if mae <= 1
              else "Juge trop eloigne de nos notes : ne pas s'y fier seul.")

A COMPLETER : renseigner MANUAL_SCORES apres avoir lu les 5 resumes.
Tant que la liste n'est pas remplie, aucun accord juge/humain n'est calcule.


## 7. Tache 3 - L'audience change-t-elle vraiment le registre ?

Le starter concluait  registre change  des que les deux textes differaient, ce qui
est vrai de n'importe quelle paire de generations et ne prouve rien. On mesure donc
le **vocabulaire reellement specifique** a chaque version.

In [9]:
source = DOCS[0]["text"]


def words(text):
    return {w.strip(".,:;()").lower() for w in text.split() if len(w) > 4}


def compare(a, b):
    """Vocabulary overlap between two versions: 100% means nothing changed."""
    return len(words(a) & words(b)) / max(len(words(a) | words(b)), 1)


# v1: the audience is only named, as in the starter prompt.
v1_manager = summarize(source, audience="a non-technical manager")
v1_engineer = summarize(source, audience="a senior software engineer")
v1_overlap = compare(v1_manager, v1_engineer)

print("=== v1 : audience simplement nommee ===")
print(f"Recouvrement du vocabulaire : {v1_overlap:.0%}")

=== v1 : audience simplement nommee ===
Recouvrement du vocabulaire : 100%


Le recouvrement mesure ci-dessus est total : nommer l'audience ne suffit pas, le
modele produit deux fois le meme texte. On formule donc une v2 qui **dit quoi
changer** plutot que **pour qui ecrire**, et on remesure a l'identique.

In [10]:
PROMPT_V2 = """Summarize the meeting transcript below.

Write for: {audience}
{register}

Hard limits:
- At most {max_words} words in total.
- Exactly 3 bullet points, each starting with "- ".
- Then a line containing exactly "Actions:" followed by one line per action item,
  each starting with "- ". Write "- none" if there are no action items.

Return nothing else.

Transcript:
{text}"""

REGISTERS = {
    "a non-technical manager":
        "Use plain business language. Ban technical jargon entirely. Emphasise"
        " impact, deadlines and risk to the customer.",
    "a senior software engineer":
        "Use precise technical vocabulary. Name the components, systems and"
        " failure modes involved. Skip business framing.",
}


def summarize_v2(text, audience, max_words=MAX_WORDS):
    return ask(
        PROMPT_V2.format(audience=audience, register=REGISTERS[audience],
                         max_words=max_words, text=text),
        temperature=TEMPERATURE,
        model=MODEL,
    )


v2_manager = summarize_v2(source, "a non-technical manager")
v2_engineer = summarize_v2(source, "a senior software engineer")
v2_overlap = compare(v2_manager, v2_engineer)

print("--- v2 manager ---\n", v2_manager)
print("\n--- v2 engineer ---\n", v2_engineer)
print(f"\nRecouvrement v1 : {v1_overlap:.0%}")
print(f"Recouvrement v2 : {v2_overlap:.0%}")
print("Specifique au manager   :", sorted(words(v2_manager) - words(v2_engineer))[:8])
print("Specifique a l'ingenieur:", sorted(words(v2_engineer) - words(v2_manager))[:8])
print("\nVERDICT:", "v2 differencie reellement le registre" if v2_overlap < v1_overlap - 0.1
      else "meme avec des consignes explicites, le modele 3B ne change pas de registre")

--- v2 manager ---
 - The new mobile checkout redesign has reduced the number of fields from nine to five.
- However, accessibility testing revealed a problem with screen readers not announcing validation errors.
- This issue must be fixed before the beta release next Friday to ensure a smooth customer experience.

Actions:
- - Fix the issue with screen reader announcements before the beta release.
- - Review and test the redesign to ensure it meets accessibility standards.
- - Provide a plan for addressing any other potential issues that may arise.

--- v2 engineer ---
 - Component: Mobile checkout redesign
- System: Payment form and accessibility testing
- Failure mode: Inadequate error announcement for screen readers

- 
- 
- 

Actions:
- Implement ARIA attributes for validation error announcements
- Conduct accessibility testing for screen readers
- Integrate error announcement functionality before beta release

Recouvrement v1 : 100%
Recouvrement v2 : 24%
Specifique au manager   :

## 8. Bonus - Les decisions extraites sont-elles les bonnes ?

Le jeu de donnees fournit une liste `action_items` de reference que ni le starter ni
la consigne n'exploitaient. C'est pourtant la sortie la plus utile du resumeur : on
mesure le rappel, en considerant qu'un item de reference est retrouve si ses mots
porteurs apparaissent dans l'un des items generes.

In [11]:
STOP = {"the", "and", "for", "with", "that", "this", "from", "will", "team", "into"}


def keywords(s):
    return {w.strip(".,:;()").lower() for w in s.split()
            if len(w) > 3 and w.strip(".,:;()").lower() not in STOP}


def matches(expected_item, produced_item):
    """A reference item counts as found only if HALF its keywords reappear.

    A looser threshold (one shared word) scored 100% on every document, which said
    more about the metric than about the model - so it is deliberately strict here.
    """
    want = keywords(expected_item)
    if not want:
        return False
    return len(want & keywords(produced_item)) / len(want) >= 0.5


def recall(expected, produced):
    """Share of reference action items recognisable in the generated ones."""
    if not expected:
        return None
    found = sum(any(matches(item, got) for got in produced) for item in expected)
    return found / len(expected)


scores = []
for i, (doc, summary) in enumerate(zip(DOCS, SUMMARIES), 1):
    _, produced, _ = parse_summary(summary)
    r = recall(doc["action_items"], produced)
    scores.append(r)
    label = "n/a" if r is None else f"{r:.0%}"
    print(f"[{i}] attendus={len(doc['action_items'])} produits={len(produced)} rappel={label}")

valid = [s for s in scores if s is not None]
if valid:
    print(f"\nRappel moyen des action items : {sum(valid) / len(valid):.0%}")

[1] attendus=1 produits=1 rappel=100%
[2] attendus=2 produits=3 rappel=100%
[3] attendus=2 produits=3 rappel=100%
[4] attendus=2 produits=3 rappel=100%
[5] attendus=1 produits=2 rappel=100%
[6] attendus=2 produits=2 rappel=50%
[7] attendus=1 produits=3 rappel=100%
[8] attendus=1 produits=2 rappel=100%

Rappel moyen des action items : 94%


## 9. Resultats et limites

**Chiffres obtenus** (llama3.2:3b, temperature 0, 8 transcriptions) :

| Mesure | Resultat | Lecture |
|---|---|---|
| Format (3 bullets + section Actions) | 100 % | Le contrat de format explicite est respecte sans exception |
| Longueur <= 50 mots | 38 % | Mediane a 54 mots : le modele deborde systematiquement d'environ 10 % |
| Juge : fidele vs falsifie | 4 contre 2 | Ecart de 2 points : le juge detecte les affirmations inventees |
| Registre, v1 (audience nommee) | 100 % de recouvrement | Nommer l'audience ne change **rien** au texte produit |
| Registre, v2 (consignes explicites) | 24 % de recouvrement | Dire *quoi changer* fonctionne la ou dire *pour qui ecrire* echoue |
| Rappel des action items | 94 % | Un seul document (le 6) tombe a 50 % |

**Les deux enseignements principaux.** D'abord, un petit modele suit tres bien une
contrainte *structurelle* (100 % sur le format) mais mal une contrainte *quantitative*
(38 % sur la longueur) : compter des mots n'est pas une operation que le decodage
effectue. Ensuite, l'echec de la v1 sur le registre est le resultat le plus utile du
projet : une consigne d'audience purement nominale est decorative, seule une consigne
operationnelle modifie la sortie.

**Limites a assumer devant le jury :**

1. **Juge et generateur sont le meme modele** (`llama3.2:3b`). Un modele est
   indulgent avec ses propres productions : le score de fidelite est donc un
   plancher optimiste, d'ou le test contradictoire de la section 6a.
2. **8 documents, c'est peu.** Les pourcentages bougent de 12,5 points des qu'un
   seul document bascule : a lire comme une tendance, pas comme une mesure fine.
3. **Le rappel des action items est lexical**, pas semantique : une reformulation
   correcte mais sans mot commun compte comme un echec, ce qui sous-estime le modele.
4. **`MANUAL_SCORES` doit etre rempli a la main** pour que la validation humaine du
   juge soit honnete. Des notes inventees invalideraient toute la section 6b.
5. **La v2 n'est pas repassee par les controles de longueur** : elle est plus verbeuse
   que la v1. Gagner sur le registre a donc coute sur la longueur - un arbitrage a
   trancher explicitement plutot qu'a masquer.